Added QWK if the score improves even if the accuracy doesnt improve too much. 

In [1]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRangePercentilesd, Resized,
    RandFlipd, RandRotate90d, RandGaussianNoised, EnsureTyped,
)
from sklearn.model_selection import train_test_split
import pandas as pd
from monai.data import DataLoader
import torch
from monai.networks.nets import ViT
import pydicom
pydicom.config.settings.reading_validation_mode = pydicom.config.WARN


In [2]:
df_data = pd.read_csv("./data/labels_duke.csv")
data = df_data.to_dict(orient="records")


train_data, val_data = train_test_split(
    data, test_size=0.2, stratify=[d["label"] for d in data], random_state=42
)

In [3]:

train_transforms = Compose([
    LoadImaged(keys="image", reader="ITKReader"),         
    EnsureChannelFirstd(keys="image"),                      
    Orientationd(keys="image", axcodes="RAS"),             
    Spacingd(keys="image", pixdim=(1.0, 1.0, 1.0),         
             mode="bilinear"),
    ScaleIntensityRangePercentilesd(                       
        keys="image", lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True),
    Resized(keys="image", spatial_size=(96, 96, 96)),      
    RandFlipd(keys="image", prob=0.5, spatial_axis=0),     
    RandRotate90d(keys="image", prob=0.5, spatial_axes=(0, 1)),
    RandGaussianNoised(keys="image", prob=0.2),
    EnsureTyped(keys="image"),
])

val_transforms = Compose([                                  
    LoadImaged(keys="image", reader="ITKReader"),
    EnsureChannelFirstd(keys="image"),
    Orientationd(keys="image", axcodes="RAS"),
    Spacingd(keys="image", pixdim=(1.0, 1.0, 1.0), mode="bilinear"),
    ScaleIntensityRangePercentilesd(keys="image", lower=1, upper=99,
                                    b_min=0.0, b_max=1.0, clip=True),
    Resized(keys="image", spatial_size=(96, 96, 96)),
    EnsureTyped(keys="image"),
])


/home/amiyazono/miniconda3/envs/monai/lib/python3.14/site-packages/monai/transforms/io/array.py:213: UserWarning: required package for reader ITKReader is not installed, or the version doesn't match requirement.
  warnings.warn(
/home/amiyazono/miniconda3/envs/monai/lib/python3.14/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


There are many times that shows an error here as runtime error transform error. It depends mainly on num_workers (deleted and check if that resolves it) and torch.multiprocessing. 

In [4]:
from monai.data import CacheDataset
import torch.multiprocessing as mp

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass
from sklearn.metrics import (
    cohen_kappa_score
)
train_ds = CacheDataset(data=train_data, transform=train_transforms, cache_rate=1.0)
val_ds   = CacheDataset(data=val_data,   transform=val_transforms,   cache_rate=1.0)




Loading dataset: 100%|██████████| 184/184 [03:33<00:00,  1.16s/it]


In [5]:
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=4, pin_memory=True)

from monai.networks.nets import DenseNet121
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 5
 
model = DenseNet121(
    spatial_dims=3,
    in_channels=1,
    out_channels=NUM_CLASSES,
    pretrained=False,
).to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)


In [6]:
from sklearn.metrics import accuracy_score, confusion_matrix
num_epochs = 200
for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        x = batch["image"].to(device)
        y = batch["label"].to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()

    # validation
    model.eval()
    correct = total = 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch["image"].to(device)
            y = batch["label"].to(device)
            outputs = model(x)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
 
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    print(f"epoch {epoch}: val acc = {accuracy_score(all_labels, all_preds)}, KWQ = {cohen_kappa_score(all_labels, all_preds, weights="quadratic")}")

epoch 0: val acc = 0.45108695652173914, KWQ = 0.003359981462171202
epoch 1: val acc = 0.5271739130434783, KWQ = 0.15782828282828287
epoch 2: val acc = 0.4891304347826087, KWQ = 0.058495821727019504
epoch 3: val acc = 0.42934782608695654, KWQ = 0.06345436036647434
epoch 4: val acc = 0.45652173913043476, KWQ = 0.011533325239771908
epoch 5: val acc = 0.47282608695652173, KWQ = 0.054458088627869605
epoch 6: val acc = 0.483695652173913, KWQ = 0.03611800385993924
epoch 7: val acc = 0.4782608695652174, KWQ = 0.04543605045436061
epoch 8: val acc = 0.45652173913043476, KWQ = 0.09182456605273615
epoch 9: val acc = 0.5108695652173914, KWQ = 0.09951060358890707
epoch 10: val acc = 0.44565217391304346, KWQ = 0.10511033681765403
epoch 11: val acc = 0.45652173913043476, KWQ = 0.06026557711950964
epoch 12: val acc = 0.44565217391304346, KWQ = 0.04809476238047672
epoch 13: val acc = 0.45108695652173914, KWQ = 0.03386198028289755
epoch 14: val acc = 0.4076086956521739, KWQ = 0.10771337288909189
epoch 15

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=74, pipe_handle=2382)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/amiyazono/miniconda3/envs/monai/lib/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/home/amiyazono/miniconda3/envs/monai/lib/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
  File "/home/amiyazono/miniconda3/envs/monai/lib/python3.14/site-packages/monai/__init__.py", line 101, in <module>
    load_submodules(sys.modules[__name__], False, exclude_pattern=excludes)
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/amiyazono/miniconda3/envs/monai/lib/python3.14/site-packages/monai/utils/module.py", line 187, in load_submodules
    mod = import_module(name)
  Fi

KeyboardInterrupt: 